# Module 4 - Week 6, Topic 1: Linear Regression
## Live Demo Notebook

**Scenario:** A Lagos estate agency wants a tool that automatically estimates property prices from key features.
We'll build a **Linear Regression** model and interpret what it learned about the Lagos property market.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style='whitegrid', palette='muted')

df = pd.read_csv('lagos_properties.csv')
print('Shape:', df.shape)
df.head()

---
## Part 1 - Explore Before Modelling

In [ ]:
print('Feature summary:')
print(df.describe().round(1))
print()
print('location_tier distribution:')
print(df['location_tier'].value_counts().sort_index())
print('  1=Mainland  2=Island midrange  3=Lekki/VI/Ikoyi')

In [ ]:
# Is the relationship between area and price roughly linear?
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Area vs price
for tier in [1, 2, 3]:
    subset = df[df['location_tier'] == tier]
    label = {1: 'Mainland', 2: 'Island mid', 3: 'Lekki/VI/Ikoyi'}[tier]
    axes[0].scatter(subset['area_sqm'], subset['price_million_naira'],
                    alpha=0.5, s=40, label=label)
axes[0].set_xlabel('Area (sqm)')
axes[0].set_ylabel('Price (NGN M)')
axes[0].set_title('Area vs Price by Location Tier')
axes[0].legend(fontsize=8)

# Correlation heatmap
corr = df.corr(numeric_only=True)
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=axes[1],
            vmin=-1, vmax=1, linewidths=0.5)
axes[1].set_title('Correlation Matrix')

plt.tight_layout()
plt.show()

print('Strongest correlation with price:', corr['price_million_naira'].sort_values(ascending=False).drop('price_million_naira'))

---
## Part 2 - Build the Linear Regression Model

In [ ]:
FEATURES = ['bedrooms', 'bathrooms', 'area_sqm', 'location_tier', 'age_years']
TARGET   = 'price_million_naira'

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  LinearRegression()),
])

pipe.fit(X_train, y_train)
print('Model trained.')

In [ ]:
y_pred = pipe.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)
mean_price = y_test.mean()

print('=== Model Evaluation ===')
print(f'MAE:  NGN {mae:.1f}M  ({mae/mean_price*100:.1f}% of average price)')
print(f'RMSE: NGN {rmse:.1f}M')
print(f'R²:   {r2:.3f}  (model explains {r2*100:.0f}% of price variation)')
print()
print(f'Average test price: NGN {mean_price:.0f}M')

---
## Part 3 - Interpret the Coefficients

In [ ]:
# Extract coefficients from the model inside the Pipeline
lr_model = pipe.named_steps['model']

coef_df = pd.DataFrame({
    'feature':     FEATURES,
    'coefficient': lr_model.coef_,
}).sort_values('coefficient', ascending=False)

print(f'Intercept: NGN {lr_model.intercept_:.1f}M')
print()
print('Coefficients (NGN M per unit of standardised feature):')
print(coef_df.to_string(index=False))

In [ ]:
# Visualise coefficients
coef_sorted = coef_df.sort_values('coefficient')
colours = ['coral' if c < 0 else 'steelblue' for c in coef_sorted['coefficient']]

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(coef_sorted['feature'], coef_sorted['coefficient'],
        color=colours, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Linear Regression Coefficients\n(Blue=positive effect, Red=negative effect)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Coefficient Value (NGN M per std unit)')
plt.tight_layout()
plt.show()

print('Reading the chart:')
print('  Longer blue bar = stronger positive effect on price')
print('  Longer red bar  = stronger negative effect on price')

In [ ]:
# Compute raw coefficients (in original feature units)
scaler  = pipe.named_steps['scaler']
raw_coefs = lr_model.coef_ / scaler.scale_   # undo standardisation

print('=== Business Interpretation (in original units) ===')
print()
for feat, raw_c in zip(FEATURES, raw_coefs):
    if feat == 'location_tier':
        print(f'  {feat:<20}: moving from tier 1 to tier 2 adds NGN {raw_c:.1f}M')
    elif feat == 'age_years':
        print(f'  {feat:<20}: each additional year of age = NGN {raw_c:.2f}M change')
    else:
        print(f'  {feat:<20}: each additional unit = NGN {raw_c:.2f}M')

---
## Part 4 - Residual Analysis

In [ ]:
residuals = y_test.values - y_pred

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Chart 1: Actual vs Predicted
axes[0].scatter(y_test, y_pred, alpha=0.6, color='steelblue', edgecolors='white', s=60)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
             color='red', linestyle='--', linewidth=2)
axes[0].set_xlabel('Actual Price (NGN M)')
axes[0].set_ylabel('Predicted Price (NGN M)')
axes[0].set_title('Actual vs Predicted')

# Chart 2: Residuals vs Predicted
axes[1].scatter(y_pred, residuals, alpha=0.6, color='coral', edgecolors='white', s=60)
axes[1].axhline(0, color='black', linestyle='--')
axes[1].set_xlabel('Predicted Price (NGN M)')
axes[1].set_ylabel('Residual (NGN M)')
axes[1].set_title('Residuals vs Predicted')

# Chart 3: Histogram of residuals
axes[2].hist(residuals, bins=20, color='mediumpurple', edgecolor='white', alpha=0.85)
axes[2].axvline(0, color='black', linestyle='--')
axes[2].set_xlabel('Residual (NGN M)')
axes[2].set_ylabel('Count')
axes[2].set_title('Distribution of Residuals')

plt.suptitle('Residual Analysis - Lagos Property Linear Regression',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Residuals: mean={residuals.mean():.2f}  std={residuals.std():.1f}')
print('Good model: residuals centred around 0, roughly symmetric distribution.')

---
## Part 5 - Predict on New Properties

In [ ]:
new_properties = pd.DataFrame({
    'bedrooms':      [2,    4,    5,    1    ],
    'bathrooms':     [1,    3,    4,    1    ],
    'area_sqm':      [75,   180,  350,  45   ],
    'location_tier': [1,    2,    3,    1    ],
    'age_years':     [5,    10,   2,    20   ],
})

predictions = pipe.predict(new_properties[FEATURES])

descriptions = [
    '2-bed flat, Mainland, 5yr',
    '4-bed, Island mid, 10yr',
    '5-bed Lekki/VI, 350sqm, 2yr',
    '1-bed studio, Mainland, 20yr',
]

print('Property Price Estimates:')
print()
for desc, pred in zip(descriptions, predictions):
    print(f'  {desc:<35}  NGN {pred:.0f}M')

---
## Summary

| Step | What we did |
|---|---|
| EDA | Scatter plots, correlation matrix |
| Model | LinearRegression inside StandardScaler Pipeline |
| Metrics | MAE, RMSE, R² - with business interpretation |
| Coefficients | Extracted and interpreted in original units |
| Residuals | Three-panel diagnostic - actual vs predicted, residuals, histogram |
| Predictions | Price estimates for four new Lagos properties |

**Next - Topic 2:** Logistic Regression. Same linear combination - but the output passes through a sigmoid function to produce a probability (0 to 1) rather than a number.